# namportfolio クイックスタート

アクティブ運用戦略の評価を、**シグナルの前処理からリスク分解まで**ひと通り
実行する例。各節は独立しているので、必要なところだけ拾って使える。

| 節 | 問い | モジュール |
|---|---|---|
| 1 | シグナルは使える状態か | `signals` |
| 2 | シグナルに予測力はあるか | `quantile` |
| 3 | ポートフォリオの中身はどうなっているか | `holdings` |
| 4 | 成績はどうだったか | `performance` |
| 5 | なぜ勝った／負けたか | `attribution` |
| 6 | リスクはどこから来ているか | `risk` |

すべての関数は **long 形式の DataFrame**（`date` / `bid` カラム＋値カラム）を受け取り、
素の DataFrame / Series を返す。描画関数は Figure を返すだけで `show()` は呼ばない。

In [ ]:
import numpy as np
import pandas as pd

import namportfolio as npf

pd.set_option("display.precision", 4)

## 0. サンプルデータ

実データの代わりに、次の性質を持つパネルを作る。

- 120 銘柄 × 60 か月
- シグナル `value` は**持続性があり、リターンへの予測力は弱い**（実際のファクターに近い）
- 先頭 12 銘柄はシグナルが欠損（カバレッジ外の銘柄を模す）
- ポートフォリオはシグナル上位に傾けた保有、ベンチマークは等ウェイト
- Barra 型のファクターエクスポージャー 5 本と、そのリターン・共分散

**列の意味**

| 列 | 内容 |
|---|---|
| `value` | シグナル（月末時点） |
| `fwd_ret_1m` / `3m` / `6m` | シグナル観測時点から先のリターン |
| `ret_1m` | その月の実現リターン（保有分析・帰属で使う） |
| `weight` / `bench_weight` | ポート／ベンチのウェイト |
| `Momentum` … `Growth` | ファクターエクスポージャー |
| `specific_risk` / `specific_ret` | 特異リスク（年率）とスペシフィックリターン |

In [ ]:
rng = np.random.default_rng(7)

DATES = pd.date_range("2020-01-31", periods=60, freq="ME")
BIDS = [f"JP{i:04d}" for i in range(120)]
SECTORS = ["Electronics", "Banks", "Pharma", "Trading", "Retail"]
FACTORS = ["Momentum", "Value", "Size", "Volatility", "Growth"]

n_assets, n_factors = len(BIDS), len(FACTORS)
sector_of = {bid: SECTORS[i % len(SECTORS)] for i, bid in enumerate(BIDS)}

# ファクター共分散（年率）。Momentum と Value は逆相関、Volatility と Growth は順相関
cov_matrix = np.diag([0.030, 0.020, 0.025, 0.050, 0.020])
cov_matrix[0, 1] = cov_matrix[1, 0] = -0.008
cov_matrix[3, 4] = cov_matrix[4, 3] = 0.010

signal = rng.normal(0, 1, n_assets)
factor_return_history, rows = [], []

for date in DATES:
    signal = 0.85 * signal + rng.normal(0, 0.5, n_assets)  # 持続性
    exposures = rng.normal(0, 1, (n_assets, n_factors))
    factor_returns = rng.multivariate_normal(
        np.zeros(n_factors), cov_matrix / 12, method="cholesky"
    )
    specific_returns = rng.normal(0, 0.25 / np.sqrt(12), n_assets)
    realised = exposures @ factor_returns + specific_returns + 0.004 * signal

    tilt = np.clip(signal, -2.0, 2.0)
    weights = np.maximum(1 / n_assets * (1 + 0.5 * tilt), 0.0)
    weights /= weights.sum()

    factor_return_history.append(pd.Series(factor_returns, index=FACTORS, name=date))
    for i, bid in enumerate(BIDS):
        row = {
            "date": date,
            "bid": bid,
            "sector": sector_of[bid],
            "value": np.nan if i < 12 else signal[i],  # 先頭 12 銘柄はカバレッジ外
            "ret_1m": realised[i],
            "weight": weights[i],
            "bench_weight": 1 / n_assets,
            "log_mktcap": rng.normal(10.0, 1.5),
            "per": rng.uniform(8.0, 32.0),
            "specific_risk": 0.25,
            "specific_ret": specific_returns[i],
        }
        row.update(dict(zip(FACTORS, exposures[i], strict=True)))
        rows.append(row)

panel = pd.DataFrame(rows)

# フォワードリターン（シグナル観測時点から先）
for horizon in (1, 3, 6):
    panel[f"fwd_ret_{horizon}m"] = panel.groupby("bid", sort=False)["ret_1m"].transform(
        lambda s, h=horizon: s.shift(-h).rolling(h).sum().shift(h - 1)
    )

# ファクターリターンと共分散は別テーブルで持つ（社内リスクモデルからの供給を想定）
factor_returns = pd.DataFrame(factor_return_history)
covariance = pd.DataFrame(
    [
        {"date": date, "factor_1": a, "factor_2": b, "cov": cov_matrix[i, j]}
        for date in DATES
        for i, a in enumerate(FACTORS)
        for j, b in enumerate(FACTORS)
    ]
)

print(f"panel: {panel.shape[0]:,} 行 × {panel.shape[1]} 列")
panel.head(3)

## 1. シグナルは使える状態か（`signals`）

分位分析にかける前に、**カバレッジの穴と外れ値**を潰しておく。ここを飛ばすと、
「特定の期間だけ分位リターンが跳ねる」といった形で後から効いてくる。

In [ ]:
coverage = npf.signals.coverage(panel, factor="value")
coverage.head(3)

In [ ]:
npf.viz.plot_coverage(coverage)

前処理は **winsorize → 標準化 → 中立化** の順に当てるのが定番。すべて `panel` と
同じ index の Series を返すので、そのまま列として代入できる。

`neutralize` はカテゴリ列（業種）を自動でダミー化するので、業種と対数時価総額を
まとめて渡せる。

In [ ]:
panel["value_w"] = npf.signals.winsorize(panel, factor="value")
panel["value_z"] = npf.signals.standardize(panel, factor="value_w")
panel["value_n"] = npf.signals.neutralize(panel, factor="value_z", by=["sector", "log_mktcap"])

npf.signals.distribution_summary(panel, factor="value").tail(3)

In [ ]:
npf.viz.plot_distribution(panel, factor="value", compare="value_w")

シグナルを複数持っているなら、合成する前に**冗長性**を見る。相関が高い 2 本を
足しても情報は増えない。

In [ ]:
correlation = npf.signals.signal_correlation(
    panel, factors=["value", "value_z", "value_n", "Momentum"]
)
npf.viz.plot_signal_correlation(correlation)

## 2. シグナルに予測力はあるか（`quantile`）

分位ポートフォリオと IC の 2 つで見る。**分位が単調に並ぶか**と、**IC が安定して
プラスか**は別の性質なので、両方確認する。

In [ ]:
quantile_returns = npf.quantile.quantile_returns(
    panel, factor="value_n", forward_return="fwd_ret_1m", n_quantiles=5
)
npf.quantile.quantile_summary(quantile_returns)

In [ ]:
npf.viz.plot_quantile_returns(quantile_returns)

分位は**順序尺度**なので、識別用の 8 色ではなく単一色相の濃淡で塗られる。
ロング・ショート（Q5 − Q1）だけは分位ではないのでアクセント色になる。

In [ ]:
npf.viz.plot_quantile_cumulative(quantile_returns)

In [ ]:
ic = npf.quantile.information_coefficient(panel, factor="value_n", forward_return="fwd_ret_1m")
npf.quantile.ic_summary(ic)

`t_stat_nw` は自己相関を補正した t 値。IC は系列相関を持つことが多く、素の `t_stat` は
有意性を過大評価する。**判断にはこちらを使う。**

In [ ]:
npf.viz.plot_ic(ic)

保有期間を延ばすと予測力がどう落ちるかを見る。回転率とのトレードオフを決める材料になる。

In [ ]:
decay = npf.quantile.factor_decay(
    panel, factor="value_n", forward_returns=["fwd_ret_1m", "fwd_ret_3m", "fwd_ret_6m"]
)
decay

In [ ]:
turnover = npf.quantile.quantile_turnover(panel, factor="value_n")
print(f"平均ターンオーバー: {turnover.mean().mean():.1%}")

npf.viz.plot_transition_matrix(npf.quantile.quantile_transition_matrix(panel, factor="value_n"))

### 分位以外の分け方

すでにクラス列を持っているなら、分位を計算せずそのまま使える。**シグナルが欠損した
銘柄群**を別クラスとして見ることもできる（欠損がランダムでないときに効く）。

In [ ]:
with_missing = npf.quantile.quantile_returns(
    panel, factor="value_n", forward_return="fwd_ret_1m", include_missing=True
)
npf.quantile.quantile_summary(with_missing)[["mean", "annualized_return", "t_stat_nw"]]

In [ ]:
# 欠損クラスは順序を持たないので、図でも濃淡に混ぜず中立色になる
npf.viz.plot_quantile_returns(with_missing)

In [ ]:
# 既存のクラス列（ここでは業種）でポートを組む場合
by_sector = npf.quantile.class_returns(panel, classes="sector", forward_return="fwd_ret_1m")
npf.viz.plot_quantile_returns(by_sector, palette="categorical")

### 2 つのシグナルの相互作用（2 次元ソート）

「このシグナルが効くのは特定の局面の銘柄だけではないか」を見る。2 本のファクターで
独立に分位を作り、その交差セルごとのリターンをヒートマップにする。

縦軸が `factor_1`、横軸が `factor_2`。

In [ ]:
cells = npf.quantile.double_sort_returns(
    panel,
    factor_1="value_n",
    factor_2="Momentum",
    forward_return="fwd_ret_1m",
    n_quantiles_1=5,
    n_quantiles_2=5,
)
matrix = npf.quantile.double_sort_summary(cells, statistic="annualized_return")
npf.viz.plot_double_sort(matrix, title="Annualised return (%)")

**リターンのヒートマップを読む前に、各セルの銘柄数を必ず確認する。** 2 本の相関が
高いと対角セルに銘柄が集まり、非対角のリターンは数銘柄の偶然になる。

（銘柄数は非負の量なので、発散配色ではなく単一色相で描かれる。）

In [ ]:
counts = npf.quantile.double_sort_counts(panel, factor_1="value_n", factor_2="Momentum")
npf.viz.plot_double_sort(counts, as_percent=False, title="Average holdings per cell")

In [ ]:
# 有意性で見たい場合は統計量を差し替える
npf.viz.plot_double_sort(
    npf.quantile.double_sort_summary(cells, statistic="t_stat"),
    as_percent=False,
    title="t-statistic",
)

## 3. ポートフォリオの中身はどうなっているか（`holdings`）

「何銘柄に、どこに賭けているか」と「**先月なぜ勝ったか**」を見る。

In [ ]:
concentration = npf.holdings.concentration(panel, top=(10, 20))
concentration.tail(3)

In [ ]:
npf.viz.plot_allocation(
    npf.holdings.allocation(panel, by="sector", benchmark_weight="bench_weight"),
    title="Active allocation",
)

寄与度分析は Brinson より手前で「どの銘柄が効いたか」に直接答える。

In [ ]:
contributors = npf.holdings.top_contributors(panel, forward_return="ret_1m", n=8)
npf.viz.plot_contribution(contributors)

In [ ]:
npf.viz.plot_turnover(npf.holdings.turnover(panel))

## 4. 成績はどうだったか（`performance`）

保有データからポートフォリオのリターン系列を作り、指標を出す。

In [ ]:
portfolio = (panel["weight"] * panel["ret_1m"]).groupby(panel["date"]).sum().rename("portfolio")
benchmark = (
    (panel["bench_weight"] * panel["ret_1m"]).groupby(panel["date"]).sum().rename("benchmark")
)

npf.performance_summary(portfolio, benchmark).to_frame("value")

In [ ]:
npf.viz.plot_cumulative_returns(portfolio, benchmark)

In [ ]:
npf.viz.plot_drawdown(portfolio, benchmark)

In [ ]:
npf.performance.drawdown_table(portfolio, top=3)

In [ ]:
npf.viz.plot_monthly_heatmap(portfolio)

ローリング指標は**尺度の違う指標を 1 枚に重ねない**（縦軸が 2 本あると、ありもしない
相関を読んでしまう）。指標ごとに図を分ける。

In [ ]:
npf.viz.plot_rolling(portfolio, benchmark, window=24, metric="information_ratio")

## 5. なぜ勝った／負けたか（`attribution`）

アクティブリターンを「どのセクターに賭けたか（配分）」と「その中で何を選んだか
（選択）」に分ける。

**期間をまたぐときはリンキングが要る。** 単期間の効果をそのまま足すと複利のぶん
合わない。`brinson_summary` は既定で Carino リンクを適用するので、`Total` 行の
`total` が幾何アクティブリターンに一致する。

In [ ]:
summary = npf.attribution.brinson_summary(
    panel, segment="sector", asset_return="ret_1m", model="bf"
)
summary

In [ ]:
totals = npf.attribution.total_returns(panel, segment="sector", asset_return="ret_1m")
geometric = (1 + totals["portfolio"]).prod() - (1 + totals["benchmark"]).prod()

print(f"幾何アクティブリターン: {geometric:.6f}")
print(f"リンク後の効果合計:     {summary.loc['Total', 'total']:.6f}")

In [ ]:
npf.viz.plot_waterfall(summary)

In [ ]:
npf.viz.plot_effects_by_segment(summary)

In [ ]:
effects = npf.attribution.brinson(panel, segment="sector", asset_return="ret_1m")
linked = npf.attribution.link(effects, totals, method="carino")

npf.viz.plot_cumulative_effects(linked)

## 6. リスクはどこから来ているか（`risk`）

エクスポージャー・共分散・特異リスクは社内リスクモデルから供給される前提で、
このパッケージは**推定を行わない**。

`benchmark_weight` を渡すと以降すべてがアクティブ基準になり、`total_risk` は
トラッキングエラーになる。

In [ ]:
exposures = npf.risk.factor_exposures(panel, factors=FACTORS, benchmark_weight="bench_weight")
npf.viz.plot_exposures(exposures)

In [ ]:
decomposition = npf.risk.risk_decomposition(
    panel, covariance, factors=FACTORS, benchmark_weight="bench_weight"
)
decomposition.tail(3)

**分散は足せるがリスクは足せない。** `factor_risk + specific_risk` は `total_risk` に
ならない（二乗和で合成される）ので、図でも積み上げない。

In [ ]:
npf.viz.plot_risk_decomposition(decomposition)

足し算で内訳を見たいときは **CCTR（リスク寄与）** を使う。こちらは全ファクター＋特異の
合計が `total_risk` に一致する。

In [ ]:
contribution = npf.risk.factor_risk_contribution(
    panel, covariance, factors=FACTORS, benchmark_weight="bench_weight"
)
last = DATES[-1]
print(f"cctr の合計: {contribution.xs(last, level=0)['cctr'].sum():.6f}")
print(f"total_risk:  {decomposition.loc[last, 'total_risk']:.6f}")

npf.viz.plot_risk_contribution(contribution)

In [ ]:
attributed = npf.risk.factor_return_attribution(
    panel,
    factor_returns,
    factors=FACTORS,
    benchmark_weight="bench_weight",
    specific_return="specific_ret",
)
npf.viz.plot_factor_contribution(attributed)

### リスクモデルは信用できるか

**バイアス統計量**は標準化リターン $z_t = r_t / \sigma_t$ の標準偏差。1 に近ければ
リスク予測が正しく、1 より大きければ**リスクを過小評価**している。

信頼区間の外に出ていれば、予測が偏っていると判断できる。

In [ ]:
active_return = attributed.groupby(level=0)["contribution"].sum()

bias = npf.risk.bias_statistic(active_return, decomposition["total_risk"])
lower, upper = npf.risk.bias_confidence_interval(len(active_return))

if bias > upper:
    verdict = "リスクを過小評価している"
elif bias < lower:
    verdict = "リスクを過大評価している"
else:
    verdict = "問題なし"

print(f"バイアス統計量: {bias:.3f}")
print(f"信頼区間 (95%): [{lower:.3f}, {upper:.3f}]")
print(f"判定: {verdict}")

In [ ]:
npf.viz.plot_bias_statistic(
    npf.risk.rolling_bias_statistic(active_return, decomposition["total_risk"], 24),
    n_periods=24,
)

## 付録: 覚えておくと役に立つこと

**入力はすべて long 形式。** `date` / `bid` カラム＋値カラム。カラム名が違う場合は
`npf.set_columns(date="trade_date", id="barra_id")` で既定を変えるか、関数ごとに
`date_col=` / `id_col=` を渡す。

**戻り値は素の DataFrame / Series。** そのまま `to_csv` や `merge` に流せる。

**描画関数は Figure を返すだけ。** `show()` は呼ばないので、保存もレポート生成も同じ形で書ける。
`ax=` を渡せば既存の Axes に描けるので、複数の図を 1 枚に組める。

```python
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(9, 8), layout="constrained")
npf.viz.plot_cumulative_returns(portfolio, benchmark, ax=axes[0])
npf.viz.plot_drawdown(portfolio, ax=axes[1])
```

**日本語ラベルを使うなら** `npf.viz.theme.use_japanese_font()` を呼ぶ。既定のフォントには
日本語グリフが無く、ラベルが豆腐になる。

**ダークテーマ**は `npf.viz.theme.set_mode("dark")`。明度を反転しただけの自動変換ではなく、
暗い背景用に選び直した配色に切り替わる。

### 検算できるところ

実装が壊れていないかを確かめる不変式がいくつかある。分析結果がおかしいと感じたら、
まずここを確認する。

| 不変式 | 意味 |
|---|---|
| 寄与度の行合計 = その期間のポートフォリオリターン | `holdings.contribution` |
| Brinson 効果の期間合計 = その期間のアクティブリターン | `attribution.brinson` |
| リンク後の総和 = 幾何アクティブリターン | `attribution.link` |
| CCTR の合計 = `total_risk` | `risk.factor_risk_contribution` |
| `factor_risk² + specific_risk² = total_risk²` | `risk.risk_decomposition` |